# Sesión 5 — Reducción de Dimensionalidad y Clustering

Código de los conceptos de la sesión y el ejercicio para practicarlos.

## 1. Aprendizaje no supervisado

Hasta ahora, en el curso hemos trabajado con **aprendizaje supervisado**:
teníamos variables de entrada $X$ y una variable objetivo (o *target*) $y$, y
el modelo aprendía a predecir $y$ a partir de $X$ (regresión, clasificación,
árboles, ensambles, etc.).

En el **aprendizaje no supervisado** la situación es distinta:

- **Solo hay entradas** ($X$): no existe una variable objetivo que el modelo
  deba predecir.
- El objetivo ya no es "predecir" sino **descubrir estructura o patrones**
  ocultos en los datos: ¿hay grupos naturales de observaciones parecidas
  entre sí? ¿se puede resumir la información en menos variables sin perder
  demasiado?
- Es un tipo de análisis **más subjetivo** que el supervisado. En supervisado
  hay un criterio de éxito relativamente objetivo (¿qué tan bien predigo
  $y$?, medido con métricas como accuracy, RMSE, etc., sobre datos que el
  modelo no vio). En no supervisado **no hay un único criterio de éxito**: no
  existe un "valor verdadero" contra el cual comparar directamente el
  resultado (con algunas excepciones construidas para fines pedagógicos,
  como el assignment de esta sesión). Distintos analistas pueden razonablemente
  elegir distintos números de grupos, distintas variables o distintas
  métricas de similitud, y todos pueden ser "correctos" bajo distintos
  criterios de negocio.

En esta sesión vemos dos grandes familias de técnicas no supervisadas:

1. **Reducción de dimensionalidad** (PCA, UMAP): resumir muchas variables en
   pocas, preservando la mayor información posible.
2. **Clustering** (jerárquico, K-Means, DBSCAN): agrupar observaciones
   parecidas entre sí.

## 2. Reducción de dimensionalidad — motivación

Muchos problemas reales tienen **decenas o cientos de variables** (alta
dimensión: 4D, 10D, 100D...). Trabajar con tantas dimensiones tiene varios
problemas: es difícil de visualizar (no podemos "ver" en 10 dimensiones),
puede haber redundancia entre variables correlacionadas, y algunos
algoritmos sufren la llamada "maldición de la dimensionalidad".

La pregunta que responde la reducción de dimensionalidad es: **¿podemos
pasar de muchas variables a pocas (2D, 1D...) preservando la mayor cantidad
de información posible?**

Hay dos estrategias generales:

1. **Eliminar variables** (selección de variables): simplemente descartar
   las que parecen menos relevantes. Es simple, pero **pierde información**
   de las variables descartadas — si una variable eliminada tenía algo de
   señal útil, esa señal se pierde por completo.
2. **Combinar / transformar variables** (extracción de variables, como
   **PCA**): en vez de descartar, se construyen **variables nuevas** que son
   combinaciones de las originales, diseñadas para que sean:
   - **Independientes/incorrelacionadas entre sí** (cada una aporta
     información distinta, sin redundancia), y
   - Ordenadas de forma que las primeras resuman la mayor cantidad posible
     de la variabilidad (información) presente en los datos originales.

   Esto permite quedarnos solo con las primeras 2 o 3 variables nuevas y
   aun así conservar la mayor parte de la información original — a
   diferencia de simplemente eliminar variables.

PCA es el método clásico de este segundo enfoque, y es el que vemos en
detalle a continuación.

## 3. PCA (Análisis de Componentes Principales)

**PCA** es un método de reducción de dimensionalidad basado en la **matriz
de varianzas-covarianzas** de los datos. La idea central es **proyectar los
datos en unas pocas direcciones (componentes) que capturen la mayor
varianza posible** — bajo el supuesto de que la varianza es un buen proxy de
"información": una dirección en la que los datos varían mucho separa bien
las observaciones, mientras que una dirección en la que casi no varían
aporta poco.

### Los componentes principales

- El **primer componente principal (PC1)** es la combinación lineal
  normalizada de las variables originales

  $$Z_1 = \phi_{11} X_1 + \phi_{12} X_2 + \dots + \phi_{1p} X_p$$

  (con $\phi_{11}^2 + \dots + \phi_{1p}^2 = 1$, la restricción de
  normalización) que **maximiza la varianza** de $Z_1$ entre todas las
  combinaciones lineales posibles de las variables originales. A los
  coeficientes $\phi$ se les llama *loadings* (cargas).
- El **segundo componente principal (PC2)** es la combinación lineal que
  maximiza la varianza **restante**, sujeta a la restricción de ser
  **ortogonal (incorrelacionada)** con PC1.
- Y así sucesivamente: cada componente siguiente maximiza la varianza
  restante, siendo ortogonal a todos los anteriores. Con $p$ variables
  originales se pueden construir hasta $p$ componentes principales, aunque
  normalmente solo nos interesan los primeros 2 o 3.

### Proporción de varianza explicada (PVE)

La proporción de varianza explicada por el componente $m$ es:

$$\text{PVE}_m = \frac{\text{varianza del componente } m}{\text{varianza total de los datos}}$$

Como los componentes son ortogonales, las PVE de todos los componentes
suman 1 (100% de la varianza total). Esto nos permite decidir cuántos
componentes retener: por ejemplo, si los primeros 2 componentes explican el
85% de la varianza total, podemos resumir los datos en 2 dimensiones
perdiendo relativamente poca información.

### Procedimiento (paso a paso)

1. **Estandarizar los datos** (media 0, varianza 1 en cada variable). Esto
   es importante porque PCA es sensible a la escala: una variable medida en
   miles dominaría artificialmente la varianza si no se estandariza.
2. Calcular la **matriz de covarianza** de los datos estandarizados.
3. Obtener los **autovectores y autovalores** de esa matriz de covarianza.
   Cada autovector define la dirección de un componente principal, y su
   autovalor asociado es la varianza explicada por ese componente.
4. **Ordenar** los autovectores de mayor a menor autovalor.
5. Elegir los **k autovectores principales** (los de mayor autovalor) según
   la varianza acumulada que se quiera retener.
6. **Proyectar** los datos originales sobre esos k autovectores para obtener
   las nuevas variables (componentes principales).

En la práctica no calculamos autovectores/autovalores a mano: usamos
`sklearn.decomposition.PCA`, que hace todo este procedimiento internamente
(usando SVD, equivalente numéricamente a la descomposición en
autovalores/autovectores de la matriz de covarianza).

### Demo: dataset `USArrests`

Este es el ejemplo clásico para introducir PCA: para cada uno de los 50
estados de EE. UU. tenemos 4 variables: `Murder`, `Assault` y `Rape` (tasas
de arrestos por cada 100.000 habitantes, por esos crímenes) y `UrbanPop`
(porcentaje de población urbana). Vamos a resumir estas 4 variables en 2
componentes principales.

In [ ]:
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.cluster.hierarchy import dendrogram, linkage
from sklearn.cluster import DBSCAN, AgglomerativeClustering, KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import adjusted_rand_score, davies_bouldin_score, silhouette_score
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore")

RANDOM_STATE = 42

# --- Paleta de colores del curso (uso consistente en todos los gráficos) ---
COLOR_CAT = [
    "#2a78d6",  # 1 azul
    "#eb6834",  # 2 naranja
    "#1baf7a",  # 3 aguamarina
    "#eda100",  # 4 amarillo
    "#e87ba4",  # 5 magenta
    "#008300",  # 6 verde
    "#4a3aa7",  # 7 violeta
    "#e34948",  # 8 rojo
]
COLOR_MUTED = "#898781"   # ejes/etiquetas secundarias
COLOR_GRID = "#e1e0d9"    # líneas de grilla
COLOR_INK = "#0b0b0b"     # texto principal
COLOR_INK_SEC = "#52514e"  # texto secundario
SURFACE = "#fcfcfb"       # fondo de las figuras

plt.rcParams.update({
    "figure.facecolor": SURFACE,
    "axes.facecolor": SURFACE,
    "savefig.facecolor": SURFACE,
    "axes.edgecolor": COLOR_MUTED,
    "axes.labelcolor": COLOR_INK_SEC,
    "axes.grid": True,
    "grid.color": COLOR_GRID,
    "grid.linewidth": 0.8,
    "text.color": COLOR_INK,
    "xtick.color": COLOR_MUTED,
    "ytick.color": COLOR_MUTED,
    "axes.prop_cycle": plt.cycler(color=COLOR_CAT),
    "axes.spines.top": False,
    "axes.spines.right": False,
    "font.size": 11,
    "figure.figsize": (7, 4.5),
})

print("Setup listo.")

In [ ]:
# Cargamos USArrests directamente desde Rdatasets (mismo dataset clásico de ISLR)
url = "https://raw.githubusercontent.com/vincentarelbundock/Rdatasets/master/csv/datasets/USArrests.csv"
arrests = pd.read_csv(url).rename(columns={"rownames": "State"}).set_index("State")
print(arrests.shape)
arrests.head()

Antes de aplicar PCA miramos rápidamente las escalas de las variables:
`Assault` está en cientos, mientras que `Murder` y `Rape` están en unidades
de un dígito o dos. Si no estandarizamos, `Assault` dominaría la varianza
total solo por su escala, no porque sea "más informativa".

In [ ]:
arrests.describe().round(1)

In [ ]:
# 1) Estandarizar (media 0, varianza 1)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(arrests.values)

# 2-6) sklearn hace internamente el cálculo de autovectores/autovalores (vía SVD)
#      y proyecta los datos sobre los componentes principales
pca = PCA(random_state=RANDOM_STATE)
scores = pca.fit_transform(X_scaled)

pve = pca.explained_variance_ratio_
print("Proporción de varianza explicada por componente:", np.round(pve, 3))
print("Varianza acumulada:", np.round(np.cumsum(pve), 3))

Con solo 2 componentes principales explicamos cerca del 87% de la
varianza total de las 4 variables originales — una reducción de 4D a 2D con
pérdida de información relativamente baja.

### Scree plot (varianza explicada por componente)

El *scree plot* grafica la varianza explicada (individual y acumulada) por
cada componente. Sirve para decidir cuántos componentes retener: se busca el
"codo" donde agregar un componente más deja de aportar mucha varianza
adicional.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4.5))

componentes = np.arange(1, len(pve) + 1)
ax.bar(componentes, pve, color=COLOR_CAT[0], width=0.5, label="Varianza explicada (individual)")
ax.plot(componentes, np.cumsum(pve), color=COLOR_CAT[1], marker="o", linewidth=2,
        label="Varianza explicada (acumulada)")

ax.set_xticks(componentes)
ax.set_xlabel("Componente principal")
ax.set_ylabel("Proporción de varianza explicada")
ax.set_ylim(0, 1.05)
ax.set_title("Scree plot — USArrests")
ax.legend(frameon=False, loc="center right")
plt.tight_layout()
plt.show()

### Biplot (PC1 vs PC2)

El **biplot** muestra en un mismo gráfico:

- La proyección de cada observación (cada estado) sobre los dos primeros
  componentes (los *scores*), y
- Las direcciones de las variables originales en ese mismo espacio (los
  *loadings*, representados como flechas): la dirección de la flecha indica
  hacia dónde crece esa variable, y su longitud indica qué tanto contribuye
  a esos dos componentes.

In [ ]:
fig, ax = plt.subplots(figsize=(8.5, 8.5))

pc1, pc2 = scores[:, 0], scores[:, 1]
ax.scatter(pc1, pc2, color=COLOR_CAT[0], alpha=0.75, s=40, zorder=3)

for estado, x, y in zip(arrests.index, pc1, pc2):
    ax.annotate(estado, (x, y), fontsize=7.5, color=COLOR_INK_SEC,
                xytext=(3, 3), textcoords="offset points")

# Loadings escalados para que se vean junto a los scores
loadings = pca.components_[:2].T  # (variables, 2)
escala = 1.15 * max(np.abs(pc1).max(), np.abs(pc2).max()) / np.abs(loadings).max()

for var, (lx, ly) in zip(arrests.columns, loadings):
    ax.annotate(
        "", xy=(lx * escala, ly * escala), xytext=(0, 0),
        arrowprops={"arrowstyle": "->", "color": COLOR_CAT[7], "linewidth": 1.8},
    )
    ax.text(lx * escala * 1.08, ly * escala * 1.08, var, color=COLOR_CAT[7],
            fontsize=11, fontweight="bold", ha="center", va="center")

ax.axhline(0, color=COLOR_GRID, linewidth=1)
ax.axvline(0, color=COLOR_GRID, linewidth=1)
ax.set_xlabel(f"PC1 ({pve[0]:.1%} de la varianza)")
ax.set_ylabel(f"PC2 ({pve[1]:.1%} de la varianza)")
ax.set_title("Biplot — USArrests (PC1 vs PC2)")
plt.tight_layout()
plt.show()

**Interpretación del biplot:** `Murder`, `Assault` y `Rape` apuntan en
direcciones parecidas y contribuyen sobre todo a PC1 — estados a la derecha
(como Florida, Nevada, California) tienen tasas altas de los tres crímenes
violentos, y estados a la izquierda (como North Dakota, Vermont) tienen
tasas bajas. `UrbanPop` apunta en una dirección bastante distinta (más
alineada con PC2), lo que indica que el porcentaje de población urbana **no
está tan correlacionado** con las tasas de criminalidad como los tres
crímenes entre sí. Es decir: **PC1 se puede leer como un eje general de
"criminalidad violenta"**, y PC2 como un eje relacionado con
urbanización, relativamente independiente del primero.

## 4. UMAP (Uniform Manifold Approximation and Projection)

> **Nota:** esta sección es puramente conceptual — no vamos a ejecutar
> código de UMAP. La librería `umap-learn` no está instalada en el entorno
> de este curso (Python 3.12/3.14), y no la vamos a instalar. Lo importante
> es que entiendas la idea y cuándo preferirla sobre PCA; si luego la
> necesitas en un proyecto propio, `pip install umap-learn` la instala sin
> problema en un entorno adecuado.

**UMAP** es otra técnica de reducción de dimensionalidad, mucho más
reciente que PCA, muy popular para visualizar datos de alta dimensión
(por ejemplo, embeddings de texto o imágenes, datos de expresión génica,
etc.) en 2D o 3D.

### Idea general

1. Toma los datos de **alta dimensión** y, para cada punto, calcula qué tan
   **similares** son sus vecinos más cercanos (construye un grafo de
   vecindarios en el espacio original).
2. Busca una representación en **baja dimensión** (2D, 3D) tal que ese mismo
   grafo de similitudes/vecindarios se preserve lo mejor posible.
3. Este proceso se resuelve con un algoritmo de optimización (**descenso de
   gradiente**) que va moviendo los puntos en el espacio de baja dimensión
   hasta que los puntos que eran vecinos cercanos en alta dimensión queden
   cercanos también en baja dimensión (y los que no lo eran, queden
   alejados).
4. Es relativamente **rápido**, incluso en datasets grandes (miles o
   millones de puntos), comparado con otras técnicas no lineales similares
   como t-SNE.

### UMAP vs. PCA — la diferencia clave

| | PCA | UMAP |
|---|---|---|
| Tipo de transformación | **Lineal** (combinaciones lineales de las variables originales) | **No lineal** |
| Qué preserva | **Varianza global** de los datos | **Estructura / vecindarios locales** |
| Componentes | Interpretables como combinación lineal de las variables originales | Las dimensiones resultantes no tienen una interpretación directa en términos de las variables originales |
| Determinismo | Determinista (mismo resultado siempre) | Depende de inicialización aleatoria y de hiperparámetros (n_neighbors, min_dist) |

Como UMAP preserva relaciones **locales** (vecindarios) en vez de varianza
**global**, en la práctica **a veces revela agrupamientos (clusters) que
PCA no separa bien**: dos grupos pueden estar muy cerca en las direcciones
de mayor varianza global (y por lo tanto casi superpuestos en un PCA 2D),
pero seguir siendo vecindarios densos y bien diferenciados a nivel local,
algo que UMAP sí puede capturar por su naturaleza no lineal.

En la práctica, un flujo de trabajo típico es: usar **PCA** primero cuando
se necesita algo rápido, determinista e interpretable (o como paso previo
para reducir ruido antes de aplicar otro algoritmo), y usar **UMAP** cuando
el objetivo es principalmente **visualizar** estructura compleja / no
lineal en los datos, aceptando que las nuevas dimensiones ya no son
directamente interpretables.

## 5. Motivación de clustering (casos de negocio)

El **análisis de cluster** agrupa un conjunto de observaciones en
subconjuntos (clusters o grupos) de forma que:

- Las observaciones **dentro** de un mismo cluster se parecen entre sí
  (alta **cohesión**), y
- Las observaciones de **clusters distintos** difieren entre sí (alta
  **separación**).

### Ejemplos de negocio

- **Segmentación de clientes en mercadeo**: agrupar clientes según su
  comportamiento de compra para diseñar campañas específicas por grupo. Un
  segmento clásico es el del **"cazador de ofertas"** (*bargain hunter*):
  clientes que casi solo compran cuando hay descuentos, sensibles al
  precio, con bajo valor de vida (LTV) fuera de promociones — a este
  segmento le conviene ofrecerle promociones dirigidas en vez de
  descuentos generales que erosionan el margen con clientes que habrían
  comprado de todas formas.
- **Ubicación de tiendas / sucursales** según variables demográficas de
  cada zona (ingreso promedio, densidad poblacional, edad promedio, etc.):
  agrupar zonas parecidas ayuda a decidir dónde abrir una tienda nueva
  basándose en zonas similares donde ya se sabe que el negocio funciona
  bien.

### Las dos preguntas centrales del clustering

1. **¿Cómo medimos distancia o similitud** entre dos observaciones? (Por
   ejemplo: distancia euclidiana, distancia de Manhattan, correlación,
   etc. — y **estandarizar las variables antes** suele ser crítico, igual
   que en PCA, para que una variable con escala grande no domine la
   distancia solo por su magnitud.)
2. **¿Cuántos grupos (clusters) considerar?** A diferencia de la
   clasificación supervisada, aquí normalmente no sabemos de antemano
   cuántos grupos "existen" — es algo que hay que decidir con ayuda de
   heurísticas (dendrograma, método del codo, coeficiente de silueta, o
   simplemente criterio de negocio: ¿cuántos segmentos son accionables?).

## 6. Clustering jerárquico

El clustering jerárquico construye una jerarquía completa de agrupamientos,
representada como un **dendrograma**: un árbol donde las hojas son las
observaciones individuales, y a medida que subimos en el árbol se van
fusionando en grupos cada vez más grandes, hasta llegar a un único grupo
que contiene todas las observaciones.

Una gran ventaja práctica: **no hay que fijar el número de clusters de
antemano**. Se construye el dendrograma completo una sola vez, y luego se
"corta" a la altura que se quiera para obtener 2, 3, 5... clusters —
cortar más abajo da más clusters (más específicos), cortar más arriba da
menos clusters (más generales).

### ¿Cómo se decide qué fusionar en cada paso? (criterios de enlace / *linkage*)

El algoritmo (aglomerativo, el más común) empieza con cada observación como
su propio cluster, y en cada paso fusiona los **dos clusters más
parecidos**, hasta quedar con uno solo. Para comparar la distancia entre
dos *clusters* (no solo entre dos observaciones individuales) existen
varios criterios de enlace:

- **Enlace simple / vecino más cercano (*single linkage*)**: distancia
  entre los dos clusters = la distancia **mínima** entre cualquier par de
  puntos, uno de cada cluster. Tiende a producir clusters alargados
  ("efecto cadena").
- **Enlace completo / vecino más lejano (*complete linkage*)**: distancia
  entre los dos clusters = la distancia **máxima** entre cualquier par de
  puntos. Tiende a producir clusters más compactos y balanceados.
- **Enlace promedio (*average linkage*)**: distancia = el **promedio** de
  las distancias entre todos los pares de puntos (uno de cada cluster). Un
  punto intermedio entre single y complete.
- **Método del centroide**: distancia entre los **centroides** (el punto
  promedio) de cada cluster.
- **Método de la mediana**: parecido al centroide, pero usando la mediana
  en vez del promedio — más robusto a outliers.
- (`scikit-learn`/`scipy` también ofrecen el método de **Ward**, que fusiona
  los clusters que produzcan el menor incremento en la varianza dentro de
  los grupos — es el criterio por defecto más usado en la práctica junto
  con *average* y *complete*.)

### Demo: dendrograma con `Wholesale customers`

Cargamos el dataset de clientes mayoristas (UCI), que usaremos durante
el resto de la sesión para los ejemplos de clustering. Cada fila es un
cliente, con su gasto anual (en unidades monetarias) en 6 categorías de
producto: `Fresh`, `Milk`, `Grocery`, `Frozen`, `Detergents_Paper` y
`Delicatessen`. También trae dos variables categóricas: `Channel` (canal
del cliente: Horeca = hotel/restaurante/café, o Retail) y `Region`
(Lisboa, Oporto u otra región de Portugal).

In [ ]:
from sklearn.datasets import fetch_openml

d = fetch_openml("wholesale-customers", version=1, as_frame=True, parser="auto")
wholesale = d.frame.copy()

# El dataset en OpenML trae V1..V7 + Channel + Region. Verificamos contra
# las estadísticas descriptivas del dataset original (d.DESCR) que:
#   V1 duplica a Region en forma numérica (se descarta)
#   V2..V7 son, en orden: Fresh, Milk, Grocery, Frozen, Detergents_Paper, Delicatessen
wholesale = wholesale.rename(columns={
    "V2": "Fresh", "V3": "Milk", "V4": "Grocery",
    "V5": "Frozen", "V6": "Detergents_Paper", "V7": "Delicatessen",
}).drop(columns=["V1"])

# Etiquetas legibles para las variables categóricas (para interpretar clusters más adelante)
wholesale["Channel_label"] = wholesale["Channel"].astype(int).map({1: "Horeca", 2: "Retail"})
wholesale["Region_label"] = wholesale["Region"].astype(int).map({1: "Lisboa", 2: "Oporto", 3: "Otra"})

spend_cols = ["Fresh", "Milk", "Grocery", "Frozen", "Detergents_Paper", "Delicatessen"]

print(wholesale.shape)
wholesale.head()

In [ ]:
wholesale[spend_cols].describe().round(1)

Con 440 clientes, un dendrograma de todo el dataset sería ilegible (440
hojas apretadas). Para que la demo sea visualmente clara, tomamos una
**muestra de 35 clientes** solo para esta visualización del dendrograma
(el resto de los análisis de esta sesión sí usan las 440 filas).

In [ ]:
muestra = wholesale.sample(35, random_state=RANDOM_STATE)

# Estandarizamos las variables de gasto de la muestra antes de calcular distancias
X_muestra = StandardScaler().fit_transform(muestra[spend_cols].values)

# Etiquetas de las hojas: id de fila + canal, para poder interpretar el árbol
etiquetas_hojas = [f"{idx} ({canal})" for idx, canal in zip(muestra.index, muestra["Channel_label"])]

# linkage calcula la jerarquía de fusiones; usamos "average" (enlace promedio) sobre distancia euclidiana
Z = linkage(X_muestra, method="average", metric="euclidean")

fig, ax = plt.subplots(figsize=(11, 6))
dendrogram(
    Z,
    labels=etiquetas_hojas,
    leaf_rotation=90,
    leaf_font_size=8,
    color_threshold=0.7 * max(Z[:, 2]),
    above_threshold_color=COLOR_MUTED,
    ax=ax,
)
ax.set_title("Dendrograma — muestra de 35 clientes (enlace promedio)")
ax.set_xlabel("Cliente (id, canal)")
ax.set_ylabel("Distancia (enlace promedio)")
plt.tight_layout()
plt.show()

**Cómo leerlo:** cada fusión (unión de dos ramas) ocurre a una altura que
representa qué tan distintos eran los dos clusters que se fusionaron.
Cortando el árbol horizontalmente a una altura dada obtenemos un número de
clusters: cortar más abajo (cerca de las hojas) da más clusters pequeños y
específicos, cortar más arriba da menos clusters, más generales. Los
colores marcan los clusters que resultarían de cortar en el umbral que
elegimos arriba (`color_threshold`).

## 7. K-Means

**K-Means** es el algoritmo de clustering más usado en la práctica. A
diferencia del clustering jerárquico, **sí requiere fijar de antemano** el
número de clusters $k$.

### Objetivo

K-Means busca minimizar la **variación dentro de cada cluster**, medida
como la suma de las distancias al cuadrado de cada punto al centroide de
su cluster (la *inercia*):

$$\text{Inercia} = \sum_{k=1}^{K} \sum_{i \in \text{cluster } k} \lVert x_i - \mu_k \rVert^2$$

donde $\mu_k$ es el centroide (promedio) del cluster $k$.

### Procedimiento iterativo

1. **Inicializar** $k$ centros (al azar, o con una estrategia más
   inteligente — ver `k-means++` abajo).
2. **Asignar** cada punto al centro más cercano (por distancia euclidiana).
3. **Recalcular** cada centroide como el promedio de los puntos asignados a
   él.
4. **Repetir** los pasos 2-3 hasta que las asignaciones dejen de cambiar (o
   se alcance un número máximo de iteraciones) — es decir, hasta
   converger.

### El problema de la inicialización, y `k-means++`

K-Means converge a un **mínimo local**, no necesariamente al óptimo global:
el resultado final es **sensible a la elección inicial de los centros**.
Con mala suerte en la inicialización aleatoria, se puede terminar en una
solución subóptima (clusters mal formados). La mejora estándar para
mitigar esto es **`k-means++`**: en vez de elegir los $k$ centros iniciales
completamente al azar, los elige de forma que tiendan a estar **alejados
entre sí** (probabilísticamente, dando más peso a puntos lejanos de los
centros ya elegidos), lo que en la práctica da una convergencia más rápida
y mejores resultados. `scikit-learn` usa `k-means++` **por defecto** en
`KMeans(init="k-means++")`. Aun así, en la práctica se recomienda correr el
algoritmo varias veces con distintas inicializaciones (parámetro `n_init`)
y quedarse con la mejor (menor inercia) — también es el comportamiento por
defecto de `sklearn`.

### ¿Cómo elegir $k$?

- **Método del codo (*elbow method*)**: graficar la inercia en función de
  $k$. La inercia siempre baja al aumentar $k$ (con más clusters, cada
  punto está más cerca de "su" centroide), pero la mejora marginal se hace
  cada vez más pequeña. Se busca el "codo": el punto donde agregar un
  cluster más deja de reducir sustancialmente la inercia.
- **Coeficiente de silueta**: mide, para cada punto, qué tan bien encaja en
  su cluster comparado con el cluster vecino más cercano (lo vemos en
  detalle en la sección de métricas). Se calcula para varios valores de
  $k$ y se elige el que maximiza el promedio.

### Demo completa con `Wholesale customers`

Vamos a estandarizar las 6 variables de gasto, aplicar K-Means, decidir
$k$ con el método del codo y el coeficiente de silueta, y luego interpretar
los clusters resultantes en términos del gasto promedio por categoría.

In [ ]:
scaler_wholesale = StandardScaler()
X_wholesale = scaler_wholesale.fit_transform(wholesale[spend_cols].values)

# Método del codo: inercia para k = 1..10
inercias = []
siluetas = []
rango_k = range(1, 11)
for k in rango_k:
    km = KMeans(n_clusters=k, init="k-means++", n_init=10, random_state=RANDOM_STATE)
    km.fit(X_wholesale)
    inercias.append(km.inertia_)
    if k >= 2:
        siluetas.append(silhouette_score(X_wholesale, km.labels_))
    else:
        siluetas.append(np.nan)

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

axes[0].plot(list(rango_k), inercias, color=COLOR_CAT[0], marker="o", linewidth=2)
axes[0].set_xlabel("k (número de clusters)")
axes[0].set_ylabel("Inercia (suma de distancias² al centroide)")
axes[0].set_title("Método del codo")
axes[0].set_xticks(list(rango_k))

axes[1].plot(list(rango_k)[1:], siluetas[1:], color=COLOR_CAT[1], marker="o", linewidth=2)
axes[1].set_xlabel("k (número de clusters)")
axes[1].set_ylabel("Coeficiente de silueta promedio")
axes[1].set_title("Coeficiente de silueta")
axes[1].set_xticks(list(rango_k)[1:])

plt.tight_layout()
plt.show()

for k, s in zip(rango_k, siluetas):
    if k >= 2:
        print(f"k={k}: inercia={inercias[k-1]:.1f}, silueta={s:.3f}")

El codo es tenue (el gasto de los clientes es muy heterogéneo y con
outliers marcados, así que la inercia baja de forma bastante gradual), pero
el **coeficiente de silueta** es más claro: $k=2$ y $k=3$ quedan
prácticamente empatados como los mejores valores (silueta ≈ 0.55), y a
partir de $k=4$ la silueta cae bastante. Nos quedamos con **$k=2$**, tanto
por tener la silueta más alta como por dar la segmentación más simple de
interpretar.

In [ ]:
K_ELEGIDO = 2
kmeans_final = KMeans(n_clusters=K_ELEGIDO, init="k-means++", n_init=10, random_state=RANDOM_STATE)
wholesale["cluster_kmeans"] = kmeans_final.fit_predict(X_wholesale)

print("Tamaño de cada cluster:")
print(wholesale["cluster_kmeans"].value_counts().sort_index())

print(f"\nSilueta con k={K_ELEGIDO}: {silhouette_score(X_wholesale, wholesale['cluster_kmeans']):.3f}")

### Interpretación: perfil promedio de gasto por cluster

Miramos el gasto promedio (en las unidades originales, sin estandarizar,
para que sea interpretable en términos de negocio) por categoría, en cada
cluster.

In [ ]:
perfil = wholesale.groupby("cluster_kmeans")[spend_cols].mean().round(0)
perfil["n_clientes"] = wholesale["cluster_kmeans"].value_counts().sort_index()
perfil

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))

x = np.arange(len(spend_cols))
ancho = 0.35
for i, cluster in enumerate(sorted(wholesale["cluster_kmeans"].unique())):
    valores = perfil.loc[cluster, spend_cols].values
    ax.bar(x + i * ancho, valores, width=ancho, label=f"Cluster {cluster}", color=COLOR_CAT[i])

ax.set_xticks(x + ancho / 2)
ax.set_xticklabels(spend_cols, rotation=20)
ax.set_ylabel("Gasto anual promedio (u.m.)")
ax.set_title(f"Perfil de gasto promedio por cluster (K-Means, k={K_ELEGIDO})")
ax.legend(frameon=False)
plt.tight_layout()
plt.show()

In [ ]:
# ¿Estos clusters se parecen al canal (Channel) real del cliente, que no usamos para entrenar?
pd.crosstab(wholesale["cluster_kmeans"], wholesale["Channel_label"])

**Interpretación de negocio:** el cluster 0 tiene gasto alto en `Fresh` y
`Frozen` mucho más que en `Grocery`/`Detergents_Paper`, y coincide en su
mayoría con clientes del canal **Horeca** (hoteles, restaurantes, cafés —
tiene sentido: compran productos frescos y congelados para cocinar, no
insumos de limpieza al por mayor). El cluster 1 tiene gasto mucho más alto
en `Milk`, `Grocery` y `Detergents_Paper`, y coincide en su mayoría con
clientes del canal **Retail** (tiendas — venden abarrotes y productos de
aseo empacados). Es una buena señal: aunque no usamos la columna `Channel`
para entrenar el modelo (es no supervisado), **el clustering recuperó por
su cuenta una estructura de negocio que ya existía** en los datos.

Nota: si probamos con $k=3$ o más, K-Means tiende a separar en su propio
cluster a un puñado de clientes con gasto extremadamente alto (outliers),
en vez de encontrar un tercer segmento de negocio interesante — es un buen
ejemplo de que **K-Means es sensible a outliers** (al minimizar distancias
al cuadrado, unos pocos puntos muy alejados pueden dominar un cluster
completo). Lo vas a poder comprobar tú mismo en la sección de ejercicios.

## 8. Variantes para datos categóricos: K-Modes y K-Prototypes

K-Means funciona porque tiene sentido calcular el **promedio** de variables
numéricas para obtener un centroide, y una distancia euclidiana entre
puntos numéricos. Pero ¿qué pasa si las variables son **categóricas** (por
ejemplo, `Channel_label` con valores "Horeca"/"Retail", o `Region_label`)?
No tiene sentido "promediar" categorías.

- **K-Modes**: es la versión de K-Means para variables **puramente
  categóricas**. En vez de usar la **media** como centro de cada cluster,
  usa la **moda** (el valor más frecuente) de cada variable categórica. La
  "distancia" entre dos observaciones categóricas se mide típicamente
  contando en cuántas variables difieren (distancia de coincidencia /
  *matching dissimilarity*), en vez de una distancia euclidiana.
- **K-Prototypes**: combina ambos mundos para **datos mixtos** (numéricos +
  categóricos), que es el caso más común en la práctica (por ejemplo,
  nuestro dataset de clientes mayoristas tiene tanto gasto numérico como
  `Channel`/`Region` categóricos). Calcula una distancia combinada: la
  parte numérica se trata como en K-Means (distancia euclidiana a la
  media) y la parte categórica como en K-Modes (distancia de coincidencia a
  la moda), ponderando ambas partes con un hiperparámetro.

> No instalamos la librería `kmodes` en este curso (no es una dependencia
> del entorno), así que no la vamos a ejecutar. Pero para ilustrar la idea
> de "centro = moda" con lo que ya tenemos disponible, podemos mirar cuál
> es la **moda de las variables categóricas dentro de cada cluster** que ya
> encontramos con K-Means — esto es exactamente lo que aportaría la parte
> categórica de un K-Prototypes si hubiéramos incluido `Channel`/`Region`
> desde el principio.

In [ ]:
# Moda de cada variable categórica dentro de cada cluster de K-Means
# (esto ilustra la idea de "centro = moda", el ingrediente de K-Modes/K-Prototypes)
modas = wholesale.groupby("cluster_kmeans")[["Channel_label", "Region_label"]].agg(
    lambda serie: serie.mode().iloc[0]
)
modas

Como vimos arriba en la tabla cruzada, la moda del canal en el cluster 0
es "Horeca" y en el cluster 1 es "Retail" — si hubiéramos corrido
K-Prototypes usando gasto + canal + región desde el inicio, el "centro"
categórico de cada cluster habría sido exactamente esa moda.

## 9. DBSCAN (clustering basado en densidad)

**DBSCAN** (*Density-Based Spatial Clustering of Applications with Noise*)
identifica clusters basándose en la **densidad** de puntos, en vez de
distancia a un centroide (K-Means) o fusiones jerárquicas. Esto le da
ventajas importantes:

- Puede encontrar clusters de **forma arbitraria** (no solo "blobs"
  convexos como los que encuentra K-Means).
- Es **robusto a ruido/outliers**: los puntos que no encajan bien en
  ningún cluster quedan marcados explícitamente como **anomalías**, en vez
  de ser forzados dentro de algún cluster.
- **No requiere especificar de antemano el número de clusters** — lo
  determina la propia densidad de los datos.

### Conceptos clave

Con dos hiperparámetros, `eps` (radio) y `min_samples` (número mínimo de
vecinos):

- Un punto es un **punto núcleo (*core point*)** si tiene al menos
  `min_samples` vecinos (incluyéndose a sí mismo, según la convención)
  dentro de un radio `eps`.
- Un punto que **no** es núcleo pero está dentro del radio `eps` de un
  punto núcleo pertenece al mismo cluster que ese núcleo (punto de
  **frontera**, *border point*).
- Un punto que **no** es núcleo y **no** está cerca de ningún núcleo se
  marca como **ruido / anomalía** (etiqueta `-1` en `sklearn`), no
  pertenece a ningún cluster.
- Dos puntos núcleo que estén dentro de `eps` uno del otro quedan en el
  mismo cluster, y así se van encadenando regiones densas conectadas.

### Ventajas y desventajas

**Ventajas**: no hay que fijar $k$; encuentra formas arbitrarias; separa
outliers de forma explícita.

**Desventajas**:
- Puede no ser determinista en el sentido de que, en casos límite, un
  punto de frontera que es alcanzable desde dos clusters distintos puede
  terminar asignado a uno u otro según el **orden** en que se procesen los
  datos.
- Depende fuertemente de la **métrica de distancia** elegida (igual que
  K-Means/jerárquico) y de la elección de `eps`/`min_samples`, que no
  siempre es fácil de calibrar.
- Es **problemático cuando los clusters tienen densidades muy distintas**
  entre sí: un solo valor global de `eps` puede ser demasiado grande para
  un cluster denso (fusionándolo con otro) y demasiado pequeño para uno
  disperso (fragmentándolo o marcándolo todo como ruido).

### Demo: DBSCAN sobre `Wholesale customers` (proyectado a 2D con PCA)

Para poder **visualizar** las formas de los clusters (y para que sea más
fácil calibrar `eps`), aplicamos primero PCA a 2 componentes sobre las
variables de gasto ya estandarizadas, y corremos DBSCAN sobre esa
proyección 2D.

In [ ]:
pca_2d = PCA(n_components=2, random_state=RANDOM_STATE)
X_wholesale_2d = pca_2d.fit_transform(X_wholesale)
print("Varianza explicada por los 2 componentes:", pca_2d.explained_variance_ratio_,
      "-> total:", pca_2d.explained_variance_ratio_.sum().round(3))

dbscan = DBSCAN(eps=0.4, min_samples=4)
wholesale["cluster_dbscan"] = dbscan.fit_predict(X_wholesale_2d)

conteo = wholesale["cluster_dbscan"].value_counts().sort_index()
print("\nConteo por cluster (-1 = ruido / outliers):")
print(conteo)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6.5))

etiquetas_unicas = sorted(wholesale["cluster_dbscan"].unique())
for i, etiqueta in enumerate([e for e in etiquetas_unicas if e != -1]):
    mask = wholesale["cluster_dbscan"] == etiqueta
    ax.scatter(X_wholesale_2d[mask, 0], X_wholesale_2d[mask, 1],
               color=COLOR_CAT[i], s=35, alpha=0.8, label=f"Cluster {etiqueta}")

mask_ruido = wholesale["cluster_dbscan"] == -1
ax.scatter(X_wholesale_2d[mask_ruido, 0], X_wholesale_2d[mask_ruido, 1],
           color=COLOR_MUTED, s=35, marker="x", alpha=0.9, label="Ruido / outliers")

ax.set_xlabel("PC1")
ax.set_ylabel("PC2")
ax.set_title("DBSCAN sobre Wholesale customers (proyección PCA 2D)")
ax.legend(frameon=False)
plt.tight_layout()
plt.show()

**Comparación cualitativa con K-Means:** con `eps=0.4` y `min_samples=4`,
DBSCAN encuentra un cluster grande (la mayoría de los clientes "típicos"),
un grupo pequeño adicional, y marca explícitamente como **ruido** a los
clientes con patrones de gasto más atípicos — en vez de forzarlos dentro de
alguno de los dos clusters principales, como sí hace K-Means. Esa es la
diferencia conceptual más importante entre los dos algoritmos: K-Means
**particiona todo el dataset** (cada punto pertenece a algún cluster),
mientras que DBSCAN **permite dejar puntos sin asignar**. Comparamos las
métricas de ambos formalmente en la siguiente sección.

## 10. Métricas de evaluación de clusters

Como vimos al inicio, en clustering no hay una "verdad" contra la cual
comparar directamente el resultado (no hay $y$). Aun así, existen métricas
que evalúan qué tan buena es una partición en términos de dos ideas:

- **Cohesión**: qué tan cerca están entre sí los puntos **dentro** de un
  mismo cluster (queremos que sea alta / distancias internas bajas).
- **Separación**: qué tan lejos están los puntos de **clusters distintos**
  entre sí (queremos que sea alta).

### Coeficiente de silueta (*silhouette score*)

Para cada punto $i$, compara:

- $a(i)$: distancia promedio de $i$ a los demás puntos de **su propio**
  cluster (cohesión).
- $b(i)$: distancia promedio de $i$ a los puntos del **cluster vecino más
  cercano** (separación, tomando el mejor "otro" cluster).

$$s(i) = \frac{b(i) - a(i)}{\max(a(i), b(i))}$$

$s(i)$ está entre -1 y 1: cercano a **1** significa que el punto está bien
ubicado (mucho más cerca de su cluster que del vecino más cercano), cercano
a **0** significa que está en la frontera entre dos clusters, y negativo
significa que probablemente está mal asignado (más cerca del cluster
vecino que del propio). El **coeficiente de silueta del clustering** es el
promedio de $s(i)$ sobre todos los puntos. En `sklearn`:
`silhouette_score(X, labels)`.

### Índice de Davies-Bouldin

Mide la **similitud promedio entre cada cluster y su cluster más
parecido**, donde "similitud" combina qué tan dispersos son los clusters
(cohesión) con qué tan separados están sus centroides (separación). A
diferencia de la silueta, aquí **valores más cercanos a 0 indican una
mejor partición** (clusters compactos y bien separados entre sí). En
`sklearn`: `davies_bouldin_score(X, labels)`.

### DBCV (mención conceptual)

Tanto la silueta como Davies-Bouldin asumen implícitamente clusters con
forma más o menos convexa (tipo "blob"), por lo que no son ideales para
evaluar clusters de forma arbitraria como los que produce DBSCAN. Para eso
existe **DBCV** (*Density-Based Clustering Validation*, Moulavi et al.
2014), una métrica diseñada específicamente para validar clustering
**basado en densidad**: en vez de usar distancias directas entre puntos,
evalúa la densidad relativa dentro de cada cluster frente a la densidad
entre clusters, usando una noción de "distancia de accesibilidad mutua"
similar a la que usa el propio DBSCAN/HDBSCAN. No hay una librería de DBCV
instalada en este entorno, así que **no la implementamos aquí** — mencionala
si en un proyecto propio necesitas evaluar un resultado de DBSCAN de forma
más rigurosa que con silueta/Davies-Bouldin.

### Comparación de las soluciones que obtuvimos en esta sesión

Para comparar "manzanas con manzanas", además de K-Means y DBSCAN
calculamos también una solución de clustering **jerárquico aglomerativo**
(`AgglomerativeClustering`, con enlace *ward*) sobre las 440 filas
completas, usando el mismo $k=2$ que elegimos para K-Means.

In [ ]:
agglo = AgglomerativeClustering(n_clusters=K_ELEGIDO, linkage="ward")
wholesale["cluster_jerarquico"] = agglo.fit_predict(X_wholesale)

# Para DBSCAN, las métricas de cohesión/separación solo tienen sentido sobre los puntos
# que sí quedaron asignados a algún cluster (excluimos el ruido, etiqueta -1)
mask_no_ruido = wholesale["cluster_dbscan"] != -1

resultados = pd.DataFrame([
    {
        "solución": f"K-Means (k={K_ELEGIDO})",
        "n_clusters": wholesale["cluster_kmeans"].nunique(),
        "n_ruido": 0,
        "silueta": silhouette_score(X_wholesale, wholesale["cluster_kmeans"]),
        "davies_bouldin": davies_bouldin_score(X_wholesale, wholesale["cluster_kmeans"]),
    },
    {
        "solución": f"Jerárquico - ward (k={K_ELEGIDO})",
        "n_clusters": wholesale["cluster_jerarquico"].nunique(),
        "n_ruido": 0,
        "silueta": silhouette_score(X_wholesale, wholesale["cluster_jerarquico"]),
        "davies_bouldin": davies_bouldin_score(X_wholesale, wholesale["cluster_jerarquico"]),
    },
    {
        "solución": "DBSCAN (PCA 2D, eps=0.4, min_samples=4)",
        "n_clusters": wholesale.loc[mask_no_ruido, "cluster_dbscan"].nunique(),
        "n_ruido": int((~mask_no_ruido).sum()),
        "silueta": silhouette_score(X_wholesale_2d[mask_no_ruido], wholesale.loc[mask_no_ruido, "cluster_dbscan"]),
        "davies_bouldin": davies_bouldin_score(X_wholesale_2d[mask_no_ruido], wholesale.loc[mask_no_ruido, "cluster_dbscan"]),
    },
])
resultados.round(3)

In [ ]:
print("Tamaño de clusters — Jerárquico (ward):")
print(wholesale["cluster_jerarquico"].value_counts().sort_index())

**Cuidado al leer estas métricas:** el clustering jerárquico con enlace
*ward* obtiene una silueta *más alta* que K-Means, pero al mirar el tamaño
de sus clusters se ve que es un resultado **degenerado**: aísla un puñado
de clientes con gasto extremo en un "cluster" separado y deja casi todos
los demás clientes juntos en el otro. Una silueta alta con un cluster
enorme y otro minúsculo **no necesariamente es una segmentación útil para
el negocio** — por eso siempre hay que mirar también el tamaño y la
composición de los clusters (como hicimos con la tabla cruzada contra
`Channel`), no solo el valor de la métrica. K-Means con $k=2$ sigue siendo,
de las tres soluciones, la que da un balance más razonable entre buena
silueta y una segmentación interpretable en dos grupos de tamaño
razonable. DBSCAN, por su parte, logra una silueta comparable a la de
K-Means sobre los puntos que sí asigna a un cluster, además de separar
explícitamente ~8% de los clientes como atípicos.

## 11. Resumen de conceptos clave

- El **aprendizaje no supervisado** trabaja solo con variables de entrada
  (sin $y$), busca descubrir estructura, y es más subjetivo que el
  supervisado: no hay un único criterio de éxito.
- **PCA** reduce dimensionalidad proyectando los datos (previamente
  estandarizados) sobre las direcciones de **máxima varianza**
  (autovectores de la matriz de covarianza), ordenadas y ortogonales entre
  sí. La **proporción de varianza explicada** guía cuántos componentes
  retener.
- **UMAP** es no lineal y preserva **vecindarios locales** en vez de
  varianza global — a veces revela clusters que PCA no separa bien, pero
  sus dimensiones no son directamente interpretables.
- El **clustering** agrupa observaciones parecidas (alta cohesión, alta
  separación); requiere decidir una medida de distancia y un número de
  grupos.
- El **clustering jerárquico** construye un dendrograma completo (no
  requiere fijar $k$ de antemano) usando algún criterio de enlace (single,
  complete, average, centroide, mediana, ward).
- **K-Means** minimiza la inercia (distancia² al centroide) con un proceso
  iterativo de asignación/actualización; requiere fijar $k$ y es sensible a
  la inicialización (mitigado con `k-means++`) y a outliers. $k$ se elige
  con el método del codo y/o el coeficiente de silueta.
- **K-Modes**/**K-Prototypes** extienden la idea de K-Means a variables
  categóricas (centro = moda) o mixtas.
- **DBSCAN** agrupa por densidad, encuentra formas arbitrarias, separa
  outliers explícitamente y no requiere fijar $k$ — pero depende de
  `eps`/`min_samples`, de la métrica de distancia, y no maneja bien
  clusters con densidades muy distintas.
- Para **evaluar** un clustering sin etiquetas reales: **silueta** (más
  alto mejor, combina cohesión y separación por punto), **Davies-Bouldin**
  (más cerca de 0 mejor), y conceptualmente **DBCV** para clusters basados
  en densidad. Ninguna métrica reemplaza mirar directamente el tamaño y la
  composición de los clusters resultantes.

## Cierre del curso

Con esta sesión cerramos el contenido del curso. A lo largo de las 5
sesiones recorrimos el ciclo completo de un proyecto de Machine Learning:
fundamentos, regresión y clasificación; sesgo-varianza, validación,
árboles y ensambles; desbalance de clases y series de tiempo; sistemas de
recomendación; y hoy, reducción de dimensionalidad y clustering. Lo que
sigue (taller integrador, exposiciones y examen final) es **aplicar y
demostrar** lo aprendido, no contenido nuevo.

Buen momento para volver sobre los tres retos calificados que hayan
quedado pendientes: `assignments/prediccion-accidentalidad-poblado/`,
`assignments/demanda-bicicletas/` y `assignments/recomendador-peliculas/`.

---

# Ejercicio

Reutilizan los datos ya cargados en el notebook: `wholesale`, `X_wholesale`, `X_wholesale_2d`, `spend_cols`.

| # | Parte | Tiempo sugerido |
|---|---|---|
| 1 | Elegir $k$ con el coeficiente de silueta | 10 min |
| 2 | DBSCAN: efecto de `eps` y `min_samples` | 10 min |
| 3 | K-Means directo vs. K-Means sobre PCA (ARI) | 10 min |

Si una parte se atasca, pasen a la siguiente: valen más las tres intentadas que una perfecta.

### Ejercicio 1

Prueba distintos valores de $k$ (por ejemplo, de 2 a 8) para K-Means sobre
`X_wholesale` y elige el mejor según el **coeficiente de silueta**.
Imprime el $k$ ganador y su silueta.

In [ ]:
# TODO: probar k = 2..8 en KMeans sobre X_wholesale, calcular silhouette_score
# para cada uno, e imprimir el k con mayor silueta.

### Ejercicio 2

Prueba distintas combinaciones de `eps` y `min_samples` en DBSCAN (sobre
`X_wholesale_2d`) y observa el efecto en el **número de clusters** y en el
**número de outliers (ruido)** detectados. Arma una tabla con los
resultados para al menos 4 combinaciones distintas.

In [ ]:
# TODO: probar varias combinaciones de eps/min_samples en DBSCAN sobre X_wholesale_2d,
# y para cada una reportar: número de clusters (sin contar ruido) y número de puntos de ruido.

### Ejercicio 3

Compara la segmentación de K-Means (con el mejor $k$ que encontraste en el
Ejercicio 1) cuando se entrena **directamente sobre `X_wholesale`** (las 6
variables de gasto estandarizadas) vs. cuando se entrena **sobre una
versión reducida con PCA** (por ejemplo, a 2 o 3 componentes). ¿Cambia la
segmentación resultante? Usa el **Adjusted Rand Index (ARI)** entre las dos
particiones para cuantificar qué tan parecidas son (ARI = 1 significa
particiones idénticas, salvo por cómo se numeren los clusters; ARI ≈ 0
significa tan parecidas como una asignación al azar).

In [ ]:
# TODO: entrenar KMeans (con el mejor k) sobre X_wholesale y, por separado, sobre una
# versión de X_wholesale reducida con PCA. Comparar las dos particiones con adjusted_rand_score.